# Independent Employee Engagement Survey Model

**Case Study No. 90 — Experiment 3 (proposed final model)**

This notebook trains classifiers **only after real survey responses exist**.
The IBM Streamlit demo (`app.py`, `models/final_model.joblib`) is **not** changed here.

## What this notebook does not do

- Does not invent employee survey rows
- Does not treat the empty CSV template as collected data
- Does not invent accuracy, F1, or any other metric
- Does not claim 90%+ performance before evaluation
- Does not use engagement-question columns as predictors

## Target (student-defined, not a validated scale)

\[
\text{EngagementScore} = \text{mean}(
\text{Motivation},\ 
\text{ValuedByOrganization},\ 
\text{OrganizationalConnection},\ 
\text{ExtraEffort},\ 
\text{RecommendOrganization},\ 
\text{IntentionToStay})
\]

Levels (`pd.cut` bins `[1.00, 2.33, 3.67, 5.00]`, right-closed):

- **Low** 1.00–2.33
- **Medium** 2.34–3.67
- **High** 3.68–5.00

Primary ranking metric: **macro F1**. Always report the **majority-class baseline**.


## 0. Imports and project root


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "config.py").is_file() and (candidate / "app.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root (folder with app.py and src/).")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RANDOM_STATE,
    SURVEY_DATA_PATH,
    SURVEY_MODELS_DIR,
    SURVEY_PLOTS_DIR,
    SURVEY_REPORTS_DIR,
    SURVEY_TEMPLATE_PATH,
    TEST_SIZE,
)
from src.survey_experiment import (
    ENGAGEMENT_QUESTION_COLUMNS,
    LEVEL_COLUMN,
    LEVEL_ORDER,
    LEVEL_RULES,
    PREDICTOR_COLUMNS,
    SCORE_COLUMN,
    TARGET_DISCLAIMER,
    add_survey_engagement_target,
    can_model,
    class_distribution,
    error_analysis_table,
    gini_importance_table,
    load_ibm_experiment_comparison,
    load_survey_table,
    select_survey_features,
    train_survey_models,
    validate_survey_frame,
)

sns.set_theme(style="whitegrid")
print("Project root:", PROJECT_ROOT)
print("Survey template:", SURVEY_TEMPLATE_PATH)
print("Expected data file:", SURVEY_DATA_PATH)
print("Target rule:", LEVEL_RULES)
print(TARGET_DISCLAIMER)


## 1. Dataset loading

Reads `data/survey/employee_engagement_survey.csv` if it has rows. The header-only template is not treated as data.


In [ ]:
raw_df, survey_status = load_survey_table()
print(survey_status["message"])
print("available:", survey_status["available"], "| n_rows reported:", survey_status["n_rows"])
print("path:", survey_status["path"])
if raw_df is not None:
    print("Loaded copy shape:", raw_df.shape)
    display(raw_df.head())
else:
    print("Training and metric cells below will skip. Collect real responses first.")


## 2. Data validation

Required workplace predictors and the six engagement questions must be present. Likert fields should be 1–5.


In [ ]:
if raw_df is None:
    print("Validation skipped: no survey rows.")
    validation = None
else:
    validation = validate_survey_frame(raw_df)
    print("Rows:", validation["n_rows"], "| Columns:", validation["n_columns"])
    print("Missing required columns:", validation["missing_required_columns"] or "none")
    print("Likert values outside 1–5:", validation["likert_out_of_range_1_to_5"] or "none")
    print("Ready to build target:", validation["ready_for_target"])


## 3. Missing-value analysis


In [ ]:
if raw_df is None:
    print("Missing-value analysis skipped: no survey rows.")
else:
    missing = raw_df.isna().sum().sort_values(ascending=False)
    print("Missing cells total:", int(missing.sum()))
    display(missing.to_frame("n_missing"))
    if int(missing.sum()) == 0:
        print("This file has no missing cells. No values were imputed.")
    else:
        print("Missing answers are reported only. This notebook does not invent replacements.")


## 4. Duplicate analysis


In [ ]:
if raw_df is None:
    print("Duplicate analysis skipped: no survey rows.")
else:
    n_dup = int(raw_df.duplicated().sum())
    print("Duplicate rows:", n_dup)
    if n_dup:
        display(raw_df[raw_df.duplicated(keep=False)].head())
        print("Duplicates are shown, not silently dropped, so you can decide how to clean them.")


## 5. Engagement-score construction

The six engagement questions stay in the working table. They are dropped from X later.


In [ ]:
labeled_df = None
if raw_df is None:
    print("Target construction skipped: no survey rows.")
elif validation is None or not validation["ready_for_target"]:
    print("Target construction skipped: required columns are missing.")
else:
    labeled_df = add_survey_engagement_target(raw_df)
    print("EngagementScore = mean of:", ENGAGEMENT_QUESTION_COLUMNS)
    print(labeled_df[[*ENGAGEMENT_QUESTION_COLUMNS, SCORE_COLUMN, LEVEL_COLUMN]].head())
    print("Score min/max/mean:",
          float(labeled_df[SCORE_COLUMN].min()),
          float(labeled_df[SCORE_COLUMN].max()),
          float(labeled_df[SCORE_COLUMN].mean()))


## 6. Target distribution


In [ ]:
if labeled_df is None:
    print("Target distribution skipped: no labeled rows.")
    dist = None
else:
    dist = class_distribution(labeled_df[LEVEL_COLUMN])
    display(dist)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.histplot(labeled_df[SCORE_COLUMN], bins=12, ax=axes[0], color="#3b6d8c")
    axes[0].set_title("EngagementScore (mean of six 1–5 items)")
    axes[0].set_xlabel("EngagementScore")
    sns.countplot(data=labeled_df, x=LEVEL_COLUMN, order=LEVEL_ORDER, ax=axes[1], color="#3b6d8c")
    axes[1].set_title("EngagementLevel counts")
    fig.tight_layout()
    SURVEY_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(SURVEY_PLOTS_DIR / "target_distribution.png", dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    majority_pct = float(dist["percent"].max())
    print("Largest class share (%):", majority_pct)
    if majority_pct >= 50:
        print("Classes look imbalanced. Macro F1 is the ranking metric; also report the majority baseline.")


## 7. EDA

Charts are built from the loaded table only. No causal claims.


In [ ]:
if labeled_df is None:
    print("EDA skipped: no labeled rows.")
else:
    print("Numeric summary:")
    display(labeled_df.describe(include="number").T.head(20))
    fig, ax = plt.subplots(figsize=(7, 4))
    if "Department" in labeled_df.columns:
        ct = pd.crosstab(labeled_df["Department"], labeled_df[LEVEL_COLUMN], normalize="index") * 100
        ct = ct.reindex(columns=LEVEL_ORDER)
        ct.plot(kind="bar", ax=ax, colormap="tab20c")
        ax.set_ylabel("Percent of department")
        ax.set_title("EngagementLevel share by Department (sample description only)")
        fig.tight_layout()
        fig.savefig(SURVEY_PLOTS_DIR / "eda_department.png", dpi=140, bbox_inches="tight")
        plt.show()
    plt.close(fig)


## 8. Preprocessing

Numeric columns are scaled; categoricals are one-hot encoded. The transformer is fit on **training rows only** inside the training cell. Leakage rule: engagement questions, `EngagementScore`, and `EngagementLevel` never enter X.


In [ ]:
if labeled_df is None:
    print("Feature preview skipped.")
    X_preview = None
else:
    X_preview, dropped = select_survey_features(labeled_df)
    print("Dropped from X (leakage control / non-predictors):", dropped)
    print("Predictor columns:", list(X_preview.columns))
    print("X shape:", X_preview.shape)
    leak = [c for c in ENGAGEMENT_QUESTION_COLUMNS + [SCORE_COLUMN, LEVEL_COLUMN] if c in X_preview.columns]
    print("Leakage columns still in X (should be empty):", leak)


## 9–16. Train/test split, stratified CV, models, and hyperparameter tuning

Models: Logistic Regression, Decision Tree, Random Forest, Extra Trees, Gradient Boosting. `RandomizedSearchCV` uses **macro F1**. Split: `test_size=0.20`, `random_state=42`, stratified. Training is skipped if there are fewer than 30 rows or a class has fewer than 2 people.


In [ ]:
run = None
if labeled_df is None:
    print("Modeling skipped: no labeled survey rows. No metrics were invented.")
else:
    ok, reason = can_model(labeled_df[LEVEL_COLUMN])
    if not ok:
        print("Modeling skipped:", reason)
    else:
        print("Fitting models (HPO on train folds only)...")
        run = train_survey_models(labeled_df, save_artifacts=True)
        if run.get("skipped"):
            print("Modeling skipped:", run["reason"])
            run = None
        else:
            print("Train rows:", run["n_train"], "| Test rows:", run["n_test"], "| CV folds:", run["cv_folds"])
            print("Majority class in train:", run["majority_class"])
            print("Primary ranking metric: macro F1")
            print("Selected model:", run["best_model_name"])
            print("Saved pipelines under", SURVEY_MODELS_DIR)


## 17–21. Accuracy, macro precision, macro recall, macro F1, balanced accuracy, and majority baseline


In [ ]:
if run is None:
    print("Metrics skipped: the survey model has not been trained.")
    print("Do not report a survey-model accuracy until this cell produces a sklearn table.")
else:
    comparison = run["comparison"].copy()
    display(comparison)
    print("CV macro F1 (train folds):", run["cv_macro_f1"])
    SURVEY_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(SURVEY_REPORTS_DIR / "model_comparison.csv", index=False)


## 22. Confusion matrix


In [ ]:
if run is None:
    print("Confusion matrices skipped.")
else:
    from pathlib import Path as _P
    from IPython.display import Image, display as _display
    for name, path in run["confusion_paths"].items():
        print(name)
        _display(Image(filename=path))


## 23. Feature importance

Gini importance is **model-derived** and is not a causal HR finding. Trees only.


In [ ]:
if run is None:
    print("Feature importance skipped.")
else:
    table = gini_importance_table(run["best_pipeline"])
    if table is None:
        print(run["best_model_name"], "has no feature_importances_. Showing Random Forest if it was fitted is not done here.")
        print("The selected pipeline type does not expose Gini importance.")
    else:
        display(table.head(15))
        fig, ax = plt.subplots(figsize=(8, 5.5))
        top = table.head(15).sort_values("gini_importance")
        ax.barh(top["feature"], top["gini_importance"], color="#3b6d8c")
        ax.set_xlabel("Model-derived Gini importance")
        ax.set_title(f"Top features ({run['best_model_name']}) — not causal")
        fig.tight_layout()
        fig.savefig(SURVEY_PLOTS_DIR / "feature_importance.png", dpi=140, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        table.to_csv(SURVEY_REPORTS_DIR / "feature_importance.csv", index=False)


## 24. Error analysis


In [ ]:
if run is None:
    print("Error analysis skipped.")
else:
    errors = error_analysis_table(run["X_test"], run["y_test"], run["y_pred_best"])
    print("Test accuracy check:", float(errors['correct'].mean()))
    print("Misclassified rows:", int((~errors['correct']).sum()), "of", len(errors))
    display(errors.loc[~errors["correct"]].head(20))
    errors.to_csv(SURVEY_REPORTS_DIR / "error_analysis.csv", index=False)


## Comparison of the three experiments

**Experiment 1 — IBM HR, leakage-safe (current Streamlit demo)**  
Predictors exclude the five IBM items that define the IBM `EngagementLevel`.  
Held-out Decision Tree: accuracy **65.31%**, macro F1 **0.387**. Majority baseline accuracy **73.81%**.  
This is **not** a high-performing model. Do not describe it as 90%+.

**Experiment 2 — IBM HR with target-defining variables in X (TARGET LEAKAGE / DEMONSTRATION ONLY)**  
Putting JobInvolvement, JobSatisfaction, EnvironmentSatisfaction, RelationshipSatisfaction, and WorkLifeBalance into X lets a model reconstruct the student-defined label. Accuracy above 90% in that setting is **not** evidence that unrelated HR fields independently predict engagement. The Streamlit app does **not** use this feature set.

**Experiment 3 — Independent survey (this notebook; proposed final model)**  
Workplace/HR columns predict an independently collected six-item score. Engagement questions are excluded from X.  
**No survey accuracy is reported until real responses are collected and the cells above produce sklearn scores.**


In [ ]:
ibm_table, ibm_path = load_ibm_experiment_comparison()
print("IBM investigation table path:", ibm_path)
if ibm_table is None:
    print("Could not find reports/accuracy_investigation/comparison_table.csv.")
    print("Experiment 1 numbers in the markdown cell above come from the saved IBM experiment (65.31% DT vs 73.81% majority).")
else:
    display(ibm_table)

if run is None:
    print("\nExperiment 3: not evaluated. No survey-model accuracy to add.")
else:
    print("\nExperiment 3 test-set comparison (this run):")
    display(run["comparison"][["model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "balanced_accuracy"]])
